# D3-02 Parameterised characterization factors with `edges`

This notebook builds a small two-gas climate label for the Day 1 car pair. The point is to show that `edges` can keep one mapped inventory fixed while characterization factors are reevaluated dynamically from numeric values, symbolic expressions, or external Python functions.


## Learning goals

After this notebook, you should be able to:

- load `edges` method definitions from JSON assets
- parameterise both CH4 and N2O CFs with numeric, symbolic, and external-function rules
- iterate through a CH4 x N2O concentration grid with repeated `evaluate_cfs()` calls
- explain why the matched exchanges stay fixed while CF values and total LCIA scores move


## Background references

This notebook is inspired by three `edges` example families:

- `examples/various examples/simple_parameterized_example_1.ipynb` for symbolic parameter handling
- `examples/various examples/complex_parameterized_example_1.ipynb` for user-defined functions
- `examples/publication examples/use case 3 - GWP.ipynb` and `lcia_parameterized_gwp.json` for concentration-dependent CH4 and N2O GWP logic

The external-function cells below follow the same reduced-form structure used in the publication example, which cites the Myhre et al. style `1 / sqrt(C)` radiative-efficiency term together with IPCC-style lifetimes and CO2 impulse-response parameters.


## 1) Reuse the Day 1 car pair


In [ ]:
import json
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D  # registers 3D projection

import bw2data as bd
from edges import EdgeLCIA

pd.options.display.float_format = '{:,.4f}'.format


### Select the activities


In [ ]:
bd.projects.set_current('barcelona-rlcia-2026')
db = bd.Database('bafu')

petrol_car = next(
    act for act in db
    if all(x in act['name'].lower() for x in ('gasoline', 'passenger car', 'large'))
    and act['unit'] == 'kilometer'
    and act['location'] == 'RER'
    and 'fleet' not in act['name'].lower()
)

ng_car = next(
    act for act in db
    if all(x in act['name'].lower() for x in ('compressed gas', 'passenger car', 'medium'))
    and act['unit'] == 'kilometer'
    and act['location'] == 'RER'
    and 'fleet' not in act['name'].lower()
    and 'biomethane' not in act['name'].lower()
)

activities = {
    'Petrol car': petrol_car,
    'Natural gas car': ng_car,
}

pd.DataFrame([
    {
        'activity': label,
        'name': activity['name'],
        'location': activity['location'],
        'unit': activity['unit'],
        'reference product': activity.get('reference product'),
    }
    for label, activity in activities.items()
])


### Inspect the direct CO2, CH4, and N2O flows


In [ ]:
target_flows = {'Carbon dioxide, fossil', 'Methane, fossil', 'Dinitrogen monoxide'}

flow_rows = []
for label, activity in activities.items():
    for exc in activity.biosphere():
        if exc.input['name'] not in target_flows:
            continue
        flow_rows.append(
            {
                'activity': label,
                'supplier name': exc.input['name'],
                'amount': exc['amount'],
                'unit': exc.input['unit'],
            }
        )

pd.DataFrame(flow_rows).sort_values(['supplier name', 'activity'])


## 2) Build a documented two-gas CF model

We will keep the time horizon fixed at 100 years. The external-function method follows the same concentration-dependent GWP structure used in the `edges` publication example: gas-specific radiative efficiency decreases with `1 / sqrt(C)`, and the remaining terms depend on atmospheric lifetime and the CO2 reference response. The symbolic method keeps only that concentration term and freezes the rest at the reference state.


### 2.1 Reference concentrations and time horizon


In [ ]:
CH4_REFERENCE_PPB = 1866.0
N2O_REFERENCE_PPB = 332.0
H_REFERENCE_YEARS = 100.0

pd.DataFrame([
    {'gas': 'CH4', 'reference concentration [ppb]': CH4_REFERENCE_PPB, 'time horizon [years]': H_REFERENCE_YEARS},
    {'gas': 'N2O', 'reference concentration [ppb]': N2O_REFERENCE_PPB, 'time horizon [years]': H_REFERENCE_YEARS},
])


### 2.2 Physical constants from the `edges` GWP example


In [ ]:
M_ATM = 5.15e18
M_AIR = 28.96

M_GAS = {
    'CO2': 44.01,
    'CH4': 16.04,
    'N2O': 44.013,
}

RF_COEFF = {
    'CH4': 0.036,
    'N2O': 0.12,
}

INDIRECT_FACTOR = {
    'CH4': 1.65,
    'N2O': 1.0,
}

TAU_GAS = {
    'CH4': 11.8,
    'N2O': 109.0,
}

CO2_IRF = {
    'a0': 0.2173,
    'a': [0.2240, 0.2824, 0.2763],
    'tau': [394.4, 36.54, 4.304],
}


### 2.3 Functions for concentration-dependent GWP


In [ ]:
def convert_ppb_to_mass_rf(a_ppb, gas):
    return a_ppb * (M_ATM / M_GAS[gas]) * (M_AIR / 1e9)


def agwp_co2(horizon_years):
    integral = CO2_IRF['a0'] * horizon_years + sum(
        a * tau * (1 - np.exp(-horizon_years / tau))
        for a, tau in zip(CO2_IRF['a'], CO2_IRF['tau'])
    )
    return convert_ppb_to_mass_rf(1.37e-5, 'CO2') * integral


In [ ]:
def radiative_efficiency_concentration(gas, concentration_ppb):
    alpha = RF_COEFF[gas]
    return (alpha / (2 * np.sqrt(concentration_ppb))) * INDIRECT_FACTOR[gas]


def agwp_gas(gas, horizon_years, concentration_ppb):
    mass_based_rf = convert_ppb_to_mass_rf(
        radiative_efficiency_concentration(gas, concentration_ppb),
        gas,
    )
    tau = TAU_GAS[gas]
    return mass_based_rf * tau * (1 - np.exp(-horizon_years / tau))


def gwp_concentration_dependent(gas, horizon_years, concentration_ppb):
    return agwp_gas(gas, horizon_years, concentration_ppb) / agwp_co2(horizon_years)


### 2.4 Reference CFs at the baseline atmospheric state


In [ ]:
REFERENCE_CF = {
    'CH4': gwp_concentration_dependent('CH4', H_REFERENCE_YEARS, CH4_REFERENCE_PPB),
    'N2O': gwp_concentration_dependent('N2O', H_REFERENCE_YEARS, N2O_REFERENCE_PPB),
}

pd.DataFrame([
    {
        'gas': gas,
        'reference concentration [ppb]': CH4_REFERENCE_PPB if gas == 'CH4' else N2O_REFERENCE_PPB,
        'reference CF [kg CO2-eq / kg gas]': value,
    }
    for gas, value in REFERENCE_CF.items()
])


### 2.5 Build a CH4 x N2O concentration grid


In [ ]:
ch4_levels = [1200.0, 1500.0, CH4_REFERENCE_PPB, 2200.0, 2600.0]
n2o_levels = [280.0, 310.0, N2O_REFERENCE_PPB, 360.0, 390.0]

parameter_grid = pd.DataFrame([
    {
        'case': f'case_{index:02d}',
        'ch4_concentration_ppb': ch4,
        'n2o_concentration_ppb': n2o,
    }
    for index, (ch4, n2o) in enumerate(product(ch4_levels, n2o_levels), start=1)
]).set_index('case')

grid_parameters = {
    'concentration grid': {
        'ch4_concentration_ppb': parameter_grid['ch4_concentration_ppb'].to_dict(),
        'n2o_concentration_ppb': parameter_grid['n2o_concentration_ppb'].to_dict(),
        'ch4_reference_ppb': CH4_REFERENCE_PPB,
        'n2o_reference_ppb': N2O_REFERENCE_PPB,
        'ch4_gwp100_reference': REFERENCE_CF['CH4'],
        'n2o_gwp100_reference': REFERENCE_CF['N2O'],
        'horizon_years': H_REFERENCE_YEARS,
    }
}

parameter_grid.reset_index()


### 2.6 Load the three method definitions from JSON assets


In [ ]:
ASSETS_DIR = Path('tutorials/DAY 3/assets/d3-02')
if not ASSETS_DIR.exists():
    ASSETS_DIR = Path('assets/d3-02')

method_files = {
    'numeric': ASSETS_DIR / 'numeric_climate_method.json',
    'symbolic': ASSETS_DIR / 'symbolic_climate_method.json',
    'external': ASSETS_DIR / 'external_climate_method.json',
}

methods = {
    mode: json.loads(path.read_text())
    for mode, path in method_files.items()
}

pd.DataFrame([
    {
        'mode': mode,
        'file': path.as_posix(),
        'name': methods[mode]['name'],
        'description': methods[mode]['description'],
    }
    for mode, path in method_files.items()
])


The numeric method freezes both gases at the reference outputs above. The symbolic method keeps the `1 / sqrt(C)` concentration term. The external method calls the Python function directly, so both gases are reevaluated from the same parameter grid.


## 3) Helper functions for repeated reevaluation


In [ ]:
def filter_direct_matches(lca, activity):
    df = lca.generate_cf_table(include_unmatched=False).copy()
    mask = pd.Series(True, index=df.index)
    if 'consumer name' in df.columns:
        mask &= df['consumer name'] == activity['name']
    if 'consumer reference product' in df.columns:
        mask &= df['consumer reference product'] == activity.get('reference product')
    if 'consumer location' in df.columns:
        mask &= df['consumer location'] == activity.get('location')
    return df.loc[mask].copy()


def get_cf_value(lca, activity, supplier_name):
    df = filter_direct_matches(lca, activity)
    if df.empty or 'supplier name' not in df.columns or 'CF' not in df.columns:
        return np.nan
    values = df.loc[df['supplier name'] == supplier_name, 'CF'].dropna().unique()
    return float(values[0]) if len(values) else np.nan


In [ ]:
def prepare_lca(activity, method, parameters=None, allowed_functions=None):
    kwargs = {'demand': {activity: 1}, 'method': method}
    if parameters is not None:
        kwargs['parameters'] = parameters
        kwargs['scenario'] = 'concentration grid'
    if allowed_functions is not None:
        kwargs['allowed_functions'] = allowed_functions

    lca = EdgeLCIA(**kwargs)
    lca.lci()
    lca.map_exchanges()
    return lca


In [ ]:
def evaluate_grid(lcas, case_table):
    rows = []
    for case, values in case_table.iterrows():
        for label, activity in activities.items():
            lca = lcas[label]
            lca.evaluate_cfs(scenario_idx=case)
            lca.lcia()
            rows.append(
                {
                    'case': case,
                    'ch4_concentration_ppb': values['ch4_concentration_ppb'],
                    'n2o_concentration_ppb': values['n2o_concentration_ppb'],
                    'activity': label,
                    'methane CF': get_cf_value(lca, activity, 'Methane, fossil'),
                    'n2o CF': get_cf_value(lca, activity, 'Dinitrogen monoxide'),
                    'score': float(lca.score),
                }
            )
    return pd.DataFrame(rows)


In [ ]:
def plot_score_surfaces(results, title):
    fig = plt.figure(figsize=(12, 4.8))

    for idx, label in enumerate(activities, start=1):
        ax = fig.add_subplot(1, 2, idx, projection='3d')
        subset = results.loc[results['activity'] == label].copy()

        surface = (
            subset.pivot(index='n2o_concentration_ppb', columns='ch4_concentration_ppb', values='score')
            .sort_index()
            .sort_index(axis=1)
        )

        x_values = surface.columns.to_numpy(dtype=float)
        y_values = surface.index.to_numpy(dtype=float)
        X, Y = np.meshgrid(x_values, y_values)
        Z = surface.to_numpy()

        ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.88, linewidth=0)
        ax.scatter(
            subset['ch4_concentration_ppb'],
            subset['n2o_concentration_ppb'],
            subset['score'],
            color='black',
            s=18,
        )

        ax.set_title(label)
        ax.set_xlabel('CH4 concentration [ppb]')
        ax.set_ylabel('N2O concentration [ppb]')
        ax.set_zlabel('LCIA score [kg CO2-eq / km]')
        ax.view_init(elev=25, azim=-135)

    fig.suptitle(title, y=0.96)
    fig.subplots_adjust(top=0.86, wspace=0.10)
    plt.show()


In [ ]:
def summarize_extremes(results):
    rows = []
    for label, group in results.groupby('activity'):
        low = group.loc[group['score'].idxmin()]
        high = group.loc[group['score'].idxmax()]
        rows.append(
            {
                'activity': label,
                'lowest-score case': low['case'],
                'lowest CH4 [ppb]': low['ch4_concentration_ppb'],
                'lowest N2O [ppb]': low['n2o_concentration_ppb'],
                'lowest score': low['score'],
                'highest-score case': high['case'],
                'highest CH4 [ppb]': high['ch4_concentration_ppb'],
                'highest N2O [ppb]': high['n2o_concentration_ppb'],
                'highest score': high['score'],
            }
        )
    return pd.DataFrame(rows)


## 4) Start with the fixed numeric benchmark


In [ ]:
numeric_rows = []
for label, activity in activities.items():
    lca = prepare_lca(activity, methods['numeric'])
    lca.evaluate_cfs()
    lca.lcia()
    numeric_rows.append(
        {
            'activity': label,
            'methane CF': get_cf_value(lca, activity, 'Methane, fossil'),
            'n2o CF': get_cf_value(lca, activity, 'Dinitrogen monoxide'),
            'score': float(lca.score),
        }
    )

numeric_summary = pd.DataFrame(numeric_rows)
numeric_summary


The fixed benchmark pins both gases to one reference atmospheric state. The inventory and the matched exchanges are already in place; the later steps only change the CF values.


## 5) Sweep the symbolic CH4 and N2O approximation


In [ ]:
symbolic_lcas = {
    label: prepare_lca(activity, methods['symbolic'], parameters=grid_parameters)
    for label, activity in activities.items()
}


In [ ]:
symbolic_results = evaluate_grid(symbolic_lcas, parameter_grid)
symbolic_results.head(8)


In [ ]:
plot_score_surfaces(
    symbolic_results,
    'Symbolic reevaluation of CH4 and N2O CFs',
)


## Checkpoint 1

Use the symbolic grid to identify, for each activity, which concentration pair gives the lowest score and which gives the highest score.


In [ ]:
# TODO
# Start from `symbolic_results` and summarise the score extremes for both activities.


In [ ]:
summarize_extremes(symbolic_results)


## 6) Sweep the external concentration-dependent function


In [ ]:
allowed_functions = {
    'gwp_concentration_dependent': gwp_concentration_dependent,
}

external_lcas = {
    label: prepare_lca(
        activity,
        methods['external'],
        parameters=grid_parameters,
        allowed_functions=allowed_functions,
    )
    for label, activity in activities.items()
}


In [ ]:
external_results = evaluate_grid(external_lcas, parameter_grid)
external_results.head(8)


In [ ]:
plot_score_surfaces(
    external_results,
    'External-function reevaluation of CH4 and N2O CFs',
)


### Compare the symbolic and external gas CFs


In [ ]:
cf_comparison = (
    symbolic_results[
        ['case', 'ch4_concentration_ppb', 'n2o_concentration_ppb', 'methane CF', 'n2o CF']
    ]
    .drop_duplicates()
    .rename(columns={
        'methane CF': 'symbolic methane CF',
        'n2o CF': 'symbolic n2o CF',
    })
    .merge(
        external_results[
            ['case', 'ch4_concentration_ppb', 'n2o_concentration_ppb', 'methane CF', 'n2o CF']
        ]
        .drop_duplicates()
        .rename(columns={
            'methane CF': 'external methane CF',
            'n2o CF': 'external n2o CF',
        }),
        on=['case', 'ch4_concentration_ppb', 'n2o_concentration_ppb'],
    )
)

cf_comparison['CH4 relative difference [%]'] = (
    (cf_comparison['symbolic methane CF'] - cf_comparison['external methane CF'])
    / cf_comparison['external methane CF']
    * 100
)

cf_comparison['N2O relative difference [%]'] = (
    (cf_comparison['symbolic n2o CF'] - cf_comparison['external n2o CF'])
    / cf_comparison['external n2o CF']
    * 100
)

cf_comparison.loc[cf_comparison['case'].isin(['case_01', 'case_13', 'case_25'])]


### Compare the score ranges across the three modes


In [ ]:
mode_rows = []
for mode, results in [
    ('Numeric benchmark', numeric_summary),
    ('Symbolic grid', symbolic_results),
    ('External grid', external_results),
]:
    row = {'mode': mode}
    for label in activities:
        scores = results.loc[results['activity'] == label, 'score']
        row[f'{label} min score'] = float(scores.min())
        row[f'{label} max score'] = float(scores.max())
    mode_rows.append(row)

pd.DataFrame(mode_rows)


## Recap

After this notebook, you should now be able to:

- load `edges` method definitions from notebook-level JSON assets
- parameterise both CH4 and N2O CFs with fixed numbers, symbolic expressions, or external Python functions
- pass one CH4 x N2O parameter grid to `EdgeLCIA` and iterate over it with repeated `evaluate_cfs()` calls
- explain why one mapped inventory can lead to a moving 3D score surface when the CFs are reevaluated dynamically
- prepare for `D3-03`, where the same reevaluation pattern is extended to explicit time-dependent and scenario-dependent tables
